# 09 — Derivation: data-centre construction jobs per dollar of investment

**Purpose.** Provide a transparent, reproducible estimate of data-centre *jobs per dollar invested*,
the metric the manuscript needs to answer the equal-investment / opportunity-cost question (Work
Package A, P0). The seven deep-search runs did not yield a directly reported DC jobs-per-$ figure
that passed verification, so this notebook **derives** one from cited primary project figures and
cross-checks it two independent ways.

**Primary source.** Ryu, A. & Hiatt, S. R. (2025), *Data Center Employment Forecast Analysis*,
Hamm Institute for American Energy / USC Marshall Zage Initiative — Appendix A1 (facility-level
project examples that disclose capacity, investment, and construction/operations headcount). Each
input below carries its page and a verbatim quote.

**Boundary rules (do not blend).**
- The reported denominator is **total project investment** (includes IT/servers, power, cooling,
  land) — *not* construction-only capex. Construction is typically ~40–55% of total DC project
  cost, so construction-jobs-per-*construction*-dollar would be roughly ~2× the values here. We
  report the conservative total-investment version because that is what sources disclose.
- Construction figures are **peak on-site headcount** (a stock), operations figures are permanent
  **FTE** (a flow). They are kept separate and never added.
- "Total jobs" / blended figures (e.g. Amazon Rainier "construction and operations jobs") are used
  only as a secondary sanity check, never as a construction metric.

**What a human must confirm before citing** (per the project's verification rule): the page and
quote for each project below against the primary PDF. Inputs are provenance-tagged so any value can
be corrected and the notebook re-run.


In [1]:
import pandas as pd, json
from datetime import datetime
from pathlib import Path

OUT = Path("outputs_dc_jobs_per_dollar"); OUT.mkdir(exist_ok=True)

# ---- INPUT: cited DC projects (Hamm Institute 2025, Appendix A1) ----
# capex_type = "total_investment": whole-project cost, NOT construction-only.
rows = [
 dict(project="Meta Richland Parish (LA)", mw=2000, capex_musd=10000, capex_type="total_investment",
      constr_peak_workers=5000, ops_jobs=500, blended_jobs=None,
      source="Hamm Institute (2025), Appendix A1", page="p.8 (PDF p.9)",
      quote="2,000 MW; 5,000+ peak construction workers = 2.5 jobs/MW; 500+ operational jobs, $10B investment = 0.25 job/MW"),
 dict(project="Amazon Project Rainier (IN)", mw=2200, capex_musd=11000, capex_type="total_investment",
      constr_peak_workers=None, ops_jobs=None, blended_jobs=1000,
      source="Hamm Institute (2025), Appendix A1", page="p.8 (PDF p.9)",
      quote="2,200 MW; $11 billion project with 1,000+ construction and operations jobs (not separated; secondary)"),
 dict(project="Meta Montgomery (AL)", mw=120, capex_musd=800, capex_type="total_investment",
      constr_peak_workers=None, ops_jobs=100, blended_jobs=None,
      source="Hamm Institute (2025), Appendix A1", page="p.8 (PDF p.9)",
      quote="$800M AI-optimized data center expected to support 100 operational jobs; ~120 MW = 0.83 job/MW"),
 dict(project="Google Jackson County (AL)", mw=100, capex_musd=600, capex_type="total_investment",
      constr_peak_workers=None, ops_jobs=100, blended_jobs=None,
      source="Hamm Institute (2025), Appendix A1", page="p.8-9 (PDF p.9-10)",
      quote="$600M data center expected to create ~100 full-time jobs; ~100 MW = 1 job/MW (upper bound; may include non-O&M roles)"),
]
df = pd.DataFrame(rows)

# construction workers/MW examples WITHOUT capex (used only for the MW-bridge cross-check)
constr_mw_only = pd.DataFrame([
 dict(project="Vantage Frontier (TX)", mw=1400, constr_peak_workers=5000),   # 3.6 /MW
 dict(project="OpenAI Stargate (TX)",  mw=4500, constr_peak_workers=6000),   # 1.3 /MW
])
df

,project,mw,capex_musd,capex_type,constr_peak_workers,ops_jobs,blended_jobs,source,page,quote
0,Meta Richland Parish (LA),2000,10000,total_investment,5000.0,500.0,NaN,"Hamm Institute (2025), Appendix A1",p.8 (PDF p.9),"2,000 MW; 5,000+ peak construction workers = 2..."
1,Amazon Project Rainier (IN),2200,11000,total_investment,NaN,NaN,1000.0,"Hamm Institute (2025), Appendix A1",p.8 (PDF p.9),"2,200 MW; $11 billion project with 1,000+ cons..."
2,Meta Montgomery (AL),120,800,total_investment,NaN,100.0,NaN,"Hamm Institute (2025), Appendix A1",p.8 (PDF p.9),$800M AI-optimized data center expected to sup...
3,Google Jackson County (AL),100,600,total_investment,NaN,100.0,NaN,"Hamm Institute (2025), Appendix A1",p.8-9 (PDF p.9-10),$600M data center expected to create ~100 full...


In [2]:
# ---- ROUTE 1: direct — jobs per $M of total investment ----
df["constr_peak_per_musd"] = df["constr_peak_workers"] / df["capex_musd"]
df["ops_per_musd"]         = df["ops_jobs"]            / df["capex_musd"]
df["blended_per_musd"]     = df["blended_jobs"]        / df["capex_musd"]
df["capex_musd_per_mw"]    = df["capex_musd"]          / df["mw"]

constr = df["constr_peak_per_musd"].dropna()
ops    = df["ops_per_musd"].dropna()
print("Route 1 (per $1M total investment)")
print(f"  construction peak workers/$M : {constr.round(3).tolist()}  (n={constr.count()})")
print(f"  operations jobs/$M           : min {ops.min():.3f}  median {ops.median():.3f}  max {ops.max():.3f}  (n={ops.count()})")
print(f"  blended (Amazon, secondary)  : {df['blended_per_musd'].dropna().iloc[0]:.3f}")

Route 1 (per $1M total investment)
  construction peak workers/$M : [0.5]  (n=1)
  operations jobs/$M           : min 0.050  median 0.125  max 0.167  (n=3)
  blended (Amazon, secondary)  : 0.091


In [3]:
# ---- ROUTE 2: MW-bridge cross-check = (workers/MW) / (capex $M/MW) ----
capex_per_mw = df["capex_musd_per_mw"].dropna()
wpmw = [2.5, 3.6, 1.3]  # observed construction workers/MW (Richland, Vantage, Stargate)
lo = min(wpmw) / capex_per_mw.max()
hi = max(wpmw) / capex_per_mw.min()
mid = (sum(wpmw)/len(wpmw)) / capex_per_mw.median()
print("Route 2 (MW bridge)")
print(f"  capex $M/MW (total investment): {capex_per_mw.min():.2f}-{capex_per_mw.max():.2f}")
print(f"  construction peak workers/$M : {lo:.3f}-{hi:.3f}  (central {mid:.3f})")
print("  -> agrees with Route 1 (~0.5 workers/$M)")

Route 2 (MW bridge)
  capex $M/MW (total investment): 5.00-6.67
  construction peak workers/$M : 0.195-0.720  (central 0.448)
  -> agrees with Route 1 (~0.5 workers/$M)


In [4]:
# ---- SUMMARY + provenance out ----
summary = {
  "derivation_date": datetime.now().isoformat(timespec="seconds"),
  "primary_source": "Ryu & Hiatt (2025) Hamm Institute / USC, Data Center Employment Forecast Analysis, Appendix A1",
  "denominator": "TOTAL project investment (incl. IT/servers/power/land), NOT construction-only capex (~40-55% of total).",
  "headline": {
    "dc_construction_peak_workers_per_musd": "~0.2-0.7 (central ~0.5) per $1M total investment",
    "dc_operations_jobs_per_musd": "~0.05-0.17 (central ~0.1) per $1M total investment",
    "per_dollar_employment_cliff": "~5x fewer permanent than peak-construction jobs per $ invested",
  },
  "cross_checks": {
    "route2_mw_bridge_workers_per_musd": [round(lo,3), round(hi,3)],
    "minnesota_construction_sector_jobs_per_musd": 3.93,
    "reading": "A dollar into a data centre buys far fewer construction jobs than a dollar of general construction spending (3.93/$M), because DC cost is dominated by imported equipment, not on-site labour.",
  },
  "caveats": [
    "Denominator is total investment, not construction-only capex.",
    "Construction = peak headcount (stock); operations = permanent FTE (flow); never summed.",
    "Only Meta Richland Parish gives a clean construction figure with capex; the range leans on the MW bridge.",
    "Human must confirm each page/quote against the primary PDF before citing.",
  ],
}
df.to_csv(OUT/"dc_jobs_per_dollar_inputs_and_derivation.csv", index=False)
json.dump(summary, open(OUT/"dc_jobs_per_dollar_summary.json","w"), indent=2)
print(json.dumps(summary, indent=2))

{
  "derivation_date": "2026-07-03T11:54:21",
  "primary_source": "Ryu & Hiatt (2025) Hamm Institute / USC, Data Center Employment Forecast Analysis, Appendix A1",
  "denominator": "TOTAL project investment (incl. IT/servers/power/land), NOT construction-only capex (~40-55% of total).",
  "headline": {
    "dc_construction_peak_workers_per_musd": "~0.2-0.7 (central ~0.5) per $1M total investment",
    "dc_operations_jobs_per_musd": "~0.05-0.17 (central ~0.1) per $1M total investment",
    "per_dollar_employment_cliff": "~5x fewer permanent than peak-construction jobs per $ invested"
  },
  "cross_checks": {
    "route2_mw_bridge_workers_per_musd": [
      0.195,
      0.72
    ],
    "minnesota_construction_sector_jobs_per_musd": 3.93,
    "reading": "A dollar into a data centre buys far fewer construction jobs than a dollar of general construction spending (3.93/$M), because DC cost is dominated by imported equipment, not on-site labour."
  },
  "caveats": [
    "Denominator is total 

## Interpretation

**Result.** A data centre supports on the order of **0.5 peak construction workers per $1M of total
investment** (range ~0.2–0.7; two independent routes agree), and **~0.1 permanent jobs per $1M**
(range ~0.05–0.17). The per-dollar picture reproduces the employment cliff: the construction burst
is real but temporary, and the permanent footprint per dollar is roughly five times smaller.

**Why this matters for the equal-investment question.** The general construction sector supports
~3.93 jobs per $1M of construction output (Minnesota LBO). Even after adjusting the data-centre
figure up to a construction-only denominator (~1 worker/$M), a dollar invested in a data centre
buys far fewer construction jobs than a dollar of ordinary construction or social-infrastructure
spending — because most of a data centre's cost is imported equipment (servers, power, cooling),
not on-site labour. That is the opportunity-cost point the paper makes, now with a number behind it.

**Status.** Derived, cross-checked, and provenance-logged, but flagged for human confirmation of the
primary-source pages before it is cited in the manuscript. Present as a bounded range with the
total-investment denominator stated, not as a single point estimate.
